In [ ]:
!pip install patchify pystac_client stackstac rasterio monai
!pip install planetary_computer boto3

In [ ]:
# Import libraries
import os
import numpy as np
from patchify import patchify
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import boto3
import matplotlib.pyplot as plt
import io
import os
import requests
# For fetching sentinel satellite imagery
import pystac_client
import planetary_computer
import stackstac
import numpy as np
# To process satellite imagery
import rasterio
from torch.utils.data import Dataset
from monai.transforms import (
    Compose,
    RandFlipd,
    RandRotate90d,
    RandGaussianNoised,
    ToTensord,
    RandIntensityDistortiond
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import userdata

# Retrieve keys
aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret_key_access = userdata.get("AWS_SECRET_KEY_ACCESS")
region = "ca-central-1"

s3_client = boto3.client(
    's3',
    region_name = region,
    aws_access_key_id = aws_access_key_id,
    aws_secret_access_key = aws_secret_key_access
)

In [ ]:
# Define and create 3 buckets
prefix = "forest-carbon-dung"
bucket_names = [f"{prefix}-raw", f"{prefix}-processed", f"{prefix}-results"]

for bucket in bucket_names:
  try:
    s3_client.create_bucket(Bucket=bucket, CreateBucketConfiguration={'LocationConstraint': region})
    print(f"Bucket {bucket} created successfully!")
  except Exception as e:
    print(f"Bucket {bucket} already exists")

In [ ]:
# Verify setup
response = s3_client.list_buckets()
print("Current buckets:")
for b in response['Buckets']:
  print(f"{b['Name']}")

## Preprocessing
All refer from
1. `download_sentinel`.py file (**data folder**) - compare different bands (only red and infrared are used now - dry season?)
2. `dataset.py` file under **src/models/**
3. Tiling/Chipping -> create_and_upload_chips (modified from the original version - now connect to AWS & best for streaming data, avoid crashing the computer) `preprocess.py` file (**data folder**)

In [ ]:
## 1.
import requests
# Setup AWS Client
s3_client = boto3.client(
    's3',
    region_name = region,
    aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY')
)
# Config for 3 target regions
REGIONS_CONFIG = {
    "amazon": {
        'bbox': [-60.1, -3.2, -59.9, -3.0],
        'date_range': "2023-07-01/2023-09-30" # Best weather condition for Brazil
    },
    "vietnam": {
        "bbox": [107.5, 12.5, 107.8, 12.8],
        "date_range": "2023-12-01/2024-02-28" # Best for SE Asia - dry season
    },
    "central_africa": {
        "bbox": [11.4, -0.1, 11.7, 0.2],
        "date_range": "2023-06-01/2023-08-31" # Best for Congo Basin
    }
}

def ingest_all_regions(bucket_name):
    # Connect with Microsoft Planetary Computer
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )
    for name, config in REGIONS_CONFIG.items():
      print(f"---Fetching data & Mask for: {name.upper()}---")

      # 1. Search for Sentinel-2
      search = catalog.search(
          collections=['sentinel-2-l2a'],
          bbox=config['bbox'],
          datetime=config['date_range'],
          query={"eo:cloud_cover":{"lt":5}},
          sortby=[{"field": "properties.eo:cloud_cover", "direction": "asc"}]
      )
      items = search.item_collection()
      print(f"Found {len(items)} images satisfied!")
      if not items:
        print("No clear Sentinel-2 items matched the search criteria.")
        continue
      best_item = items[0]
      date = best_item.properties["datetime"].split("T")[0]

      # 2. Find Dynamic World (Mask)
      # Search using the same footprint and time as the satellite image
      dw_search = catalog.search(
          collections=['dynamic-world'],
          intersects=best_item.geometry,
          datetime=best_item.properties["datetime"]
      )
      dw_item = dw_search.item_collection()
      if not dw_item:
        print("No matching mask found for this item")
        continue
      dw_item = dw_item[0]

      # 2. Stream to S3 (2 bands + 1 masks - B04 (Red) and B08 (NIR) - Essential for Forest Monitoring)
      assets_to_stream = {
          "B04": f"raw/region={name}/date={date}/B04.tif",
          "B08": f"raw/region={name}/date={date}/B08.tif",
          "mask": f"raw/region={name}/masks/forest_mask.tif" # file name for easy retrieval
      }
      for asset_key, s3_key in assets_to_stream.items():
        if asset_key == "mask":
            asset_url = dw_item.assets["label"].href
        else:
            asset_url = best_item.assets[asset_key].href
        print(f"Streaming {asset_key} -> {s3_key}...")
        with requests.get(asset_url, stream=True) as r:
            r.raise_for_status()
            s3_client.upload_fileobj(r.raw, bucket_name, s3_key)

    print("\n✅ All regions ingested into S3!")

# Run it! Ensure the bucket name are correct
ingest_all_regions("forest-carbon-dung-raw")

In [ ]:
# Collection ID for Dynamic World on planetary Computer - get masks
DW_COLLECTION = "dynamic-world"

def fectch_masks_for_stac_item(item, bucket_name, region):
  """
  Finds the matching dynamic World mask for a Sentinel-2 item & stream to S3
  """
  catalog = pystac_client.Client.open(
      "https://planetarycomputer.microsoft.com/api/stac/v1",
      modifier=planetary_computer.sign_inplace)

  # Search for Dynamic World item matching the Sentinel-2 footprint and time
  search = catalog.search(
      collections=[DW_COLLECTION],
      intersects=item.geometry,
      datetime=item.properties["datetime"]
  )

  dw_items = search.item_collection()
  if not dw_items:
    print("No matching mask found for this item")
    return None

  # Get class 'label' - include information of categories (Water, Crops, Built-area, etc.)
  mask_asset_url = dw_items[0].assets["label"].href
  target_key = f"raw/region={region}/masks/{item.id}_mask.tif"

  print(f"Streaming Mask to {target_key}...")
  with requests.get(mask_asset_url, stream=True) as r:
      r.raise_for_status()
      s3_client.upload_fileobj(r.raw, bucket_name, target_key)

  return target_key

- PyTorch can load `.npy` files faster than `.tif` files during training
- Since we'll normalize the data to 0.0-1.0, we don't need the complex metadata of a TIFF anymore, only the raw numbers for the Attention U-Net
- Workflow of the "Chip factory":
1. Set up S3 client
2. Load data from S3
3. Image processing (Normalize 0-1)
4. Mask binarization (Dynamic World: Class 1 is trees) - 1: forests, 0: everything else - Binarize masks right now to free some Ram (since they only contains 2 values at the moment - instead of tons of masks like Water, Crops, etc.)
5. Tiling logic
6. Filter - check if the chip contains enough forest - if forests covers less than threshold (e.g. 1%) -> skip it - make sure the model doesn't overlearn the bad pattern (if most of the images are non-forest, model will repetitively predict non-forest for the future images)
7. Upload, save as .npy and push to S3

In [ ]:
#3. Chip factory - preprocess.py - modified (wasn't connected to AWS prev, now ✔)
import boto3
import numpy as np
from patchify import patchify
import io

def create_and_upload_chips(region, date, raw_bucket, proc_bucket, patch_size=256, forest_threshold=0.01):
  """
  Standardize function to tile images/masks and upload to S3.
  Includes binarization and forest-density filtering

  """
  s3_client = boto3.client("s3")


  # Read GEOTiff from S3
  def read_s3_tif(key):
    response = s3_client.get_object(Bucket=raw_bucket, Key=key)
    with rasterio.open(io.BytesIO(response['Body'].read())) as src:
      return src.read(1) # Read first band

  #1. Path in S3:
  prefix = f"raw/region={region}/date={date}/"
  b04_key = f"{prefix}B04.tif"
  b08_key = f"{prefix}B08.tif"
  mask_key = f"raw/region={region}/masks/forest_mask.tif"

  print(f"--- Processing Region: {region.upper()}----")
  try:
    # 2. Load data from S3
    red = read_s3_tif(b04_key)
    nir = read_s3_tif(b08_key)
    raw_mask = read_s3_tif(mask_key)

    # 3. Image processing - Stack and normalize (Divided by 10,000 as per Sentinel-2 specs)
    # Resulting shapes (2, Height, Width)
    img = np.stack([red, nir], axis=0).astype(np.float32) / 10000.0
    img = np.clip(img, 0, 1)

    # 4. Mask binarization (Dynamic World: Class 1 is Trees)
    # Convert to 1 for Forest, 0 for everything else
    binary_mask = (raw_mask == 1).astype(np.float32)

    # 5. # Chippping (Tiling)
    # Use pachify on Heigth & Width
    # Patchify expects (H, W, C) or (H, W). Do it per channel
    channels, h, w = img.shape
    n_h = h // patch_size
    n_w = w // patch_size

    print(f"Generating {n_h * n_w} chips...")

    count = 0
    skipped = 0

    for i in range(n_h):
      for j in range(n_w):
      # Extract 256x256 patch for both bands
        y, x = i * patch_size, j * patch_size

        # Extract patches
        img_chip = img[:, y:y+patch_size, x:x+patch_size]
        mask_chip = binary_mask[y:y+patch_size, x:x+patch_size]

        # 6. FILTER: Check if the chip contains enough forest
        # If forest covers less than "forest_threshold" (e.g. 1%), skip it
        forest_density = np.mean(mask_chip)
        if forest_density < forest_threshold:
          skipped += 1
          continue

        # 7. Save as .npy and push Chip to S3 Processed bucket

        # Upload image chip
        img_buffer = io.BytesIO()
        np.save(img_buffer, img_chip)
        img_buffer.seek(0)
        img_key = f"processed/region={region}/images/chip_{count}.npy"
        s3_client.upload_fileobj(img_buffer, proc_bucket, img_key)

        # Upload mask chip
        mask_buffer = io.BytesIO()
        np.save(mask_buffer, mask_chip)
        mask_buffer.seek(0)
        mask_key_out = f"processed/region={region}/masks/mask_{count}.npy"
        s3_client.upload_fileobj(mask_buffer, proc_bucket, mask_key_out)
        count += 1


    print(f"✅ Region {region}: Created {count} chips, Skipped {skipped} empty patches.")
    return count
  except Exception as e:
    print(f"❌ Error processing region {region}: {e}")
    return 0


In [ ]:
# Create chips for each region (after running ingest)
raw_bucket = "forest-carbon-dung-raw"
proc_bucket = "forest-carbon-dung-processed"

# Get regions' list and realistic datetime from S3
regions_dates = [
    ("amazon", "2023-08-15"),
    ("vietnam", "2024-01-10"),
    ("central_africa", "2023-07-20")
]

for region, date in regions_dates:
  create_and_upload_chips(region, date, raw_bucket, proc_bucket,
                          patch_size = 256, forest_threshold=0.01)

## Attention U-Net
Refer from **src/models/**
1. `attention_unet.py` file - defined the model
2. `train.py`Train the dataset

In [ ]:
#1.
import torch
import torch.nn as nn

# Attention Gate:
class AttentionGate(nn.Module):
    """
    Attention Gate (AG) filters the skip connections to focus on relevant features .
    It suppresses irrelevant background regions and highlights target objects.
    """
    def __init__(self, F_g, F_l, F_int):
        super(AttentionGate, self).__init__()
        # W_g: Linear transformation for the gating signal (from the deeper decoder layer)
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        # W_x: Linear transformation for the skip connection (from the encoder layer)
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        #psi: Calculate the Attention coefficients alpha (0 to 1)
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid() # Force the value in range (0, 1)
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        # g: gating signal, x: skip connection feature map
        g1 = self.W_g(g) # Transform gating signal
        x1 = self.W_x(x) # Transform skip connection
        psi = self.relu(g1+x1) # Combine both signals to find feature overlap
        psi = self.psi(psi) # Generate the attention mask (0 - 1)
        return x * psi # Rescale the skip connection by attention weights

class ConvBlock(nn.Module):
    """
    Double Convolutional Block: (Conv -> BatchNorm -> ReLU) * 2
    """
    def __init__(self, in_ch, out_ch):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class AttentionUNet(nn.Module):
    """
    Attention U-Net implementation for multi-spectral satellite imagery.
    Default input channels: 4 (Red, Green, BLue, NIR)
    """
    def __init__(self, img_ch=4, output_ch=1):
        super(AttentionUNet, self).__init__()

        # Maxpooling to reduce spatial dimensions by half
        self.Maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

        # ENCODER:Down-sampling path to extract multi-scale features
        self.Conv1 = ConvBlock(img_ch, 64) # Level 1: 256x256
        self.Conv2 = ConvBlock(64, 128) # Level 2: 128 x128
        self.Conv3 = ConvBlock(128, 256)  # Level 3: 64x64
        self.Conv4 = ConvBlock(256, 512) # Level 4: 32x32
        self.Conv5 = ConvBlock(512, 1024) # Level 5 (Bottleneck) 16x16

        # DECODER: Up-sampling path with Attention Gates to reconstruct tha mask
        # Level 5 & 4
        self.Up5 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.Att5 = AttentionGate(F_g = 1024, F_l=512, F_int=256) # Refine level 4 features
        self.Up_conv5 = ConvBlock(1024 + 512, 512) # Concatenate decoded + attended features

        # Level 4 to 3
        self.Up4 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.Att4 = AttentionGate(F_g=512, F_l=256,F_int=128) # Refine level 3 features
        self.Up_conv4 = ConvBlock(512 + 256, 256)

        # Level 3 to 2
        self.Up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.Att3 = AttentionGate(F_g=256, F_l=128, F_int=64) # Refine level 2 features
        self.Up_conv3 = ConvBlock(256+128, 128)

        # Level 2 to 1
        self.Up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.Att2 = AttentionGate(F_g =128, F_l=64, F_int=32) # Refine level 1 features
        self.Up_conv2 = ConvBlock(128+64, 64)

        # Final output layer: 1x1 convolution to product binary forest mask
        self.Conv_1x1 = nn.Conv2d(64, output_ch, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        # --- ENCODING PATH----
        e1 = self.Conv1(x)
        e2 = self.Maxpool(e1)
        e2 = self.Conv2(e2)
        e3 = self.Maxpool(e2)
        e3 = self.Conv3(e3)
        e4 = self.Maxpool(e3)
        e4 = self.Conv4(e4)
        e5 = self.Maxpool(e4)
        e5 = self.Conv5(e5) # Bottleneck

        # --- DECODING PATH with Attention
        d5 = self.Up5(e5) # Upsample bottle neck
        x4 = self.Att5(g=d5, x=e4)       # Attend to encoder layer 4
        d5 = torch.cat((x4, d5), dim=1)  # Concatenate refined features
        d5 = self.Up_conv5(d5)           # Double conv

        d4 = self.Up4(d5)
        x3 = self.Att4(g=d4, x=e3)
        d4 = torch.cat((x3, d4), dim=1)
        d4 = self.Up_conv4(d4)

        d3 = self.Up3(d4)
        x2 = self.Att3(g=d3, x=e2)
        d3 = torch.cat((x2, d3), dim=1)
        d3 = self.Up_conv3(d3)

        d2 = self.Up2(d3)
        x1 = self.Att2(g=d2, x=e1)
        d2 = torch.cat((x1, d2), dim=1)
        d2 = self.Up_conv2(d2)

        # Final output mapping
        out = self.Conv_1x1(d2)
        return out


- To maximize the speed and save up on cost, we'll download images from S3 to Colab local memory **1 TIME ONLY** before traninig, instead of reading each file from S3 during training (each GET REQUESTS will slow down the speed and very costly)
- **Strategy:** We'll train on 10k-20k of images to get a base IoU score and make sure it's a good one before starting scaling up.
- download_training_data(): Delivery function - brings 20k of chips from Warehouse to our Google Drive
* Run once at the start of the session
* Don't skip, otherwise, the model will get new files from AWS and the training time will be 10x slower -> bill will explode 💸
- prepare_data_loader(): Once they're in, let the GPU train!


In [ ]:
from torch.utils.data import Dataset
import numpy as np
import torch
from monai.transforms import (
    Compose,
    RandFlipd,
    RandRotate90d,
    RandGaussianNoised,
    ToTensord,
    RandIntensityDistortiond
)

class ForestCarbonDataset(Dataset):
    """
    Custom Dataset for Loading Sentinel-2 chips and forest masks.
    Expect chpis in shape (2, 256, 256) -> Red, NIR
    """
    def __init__(self, image_files, mask_files, transform=None):
        self.image_files = image_files
        self.mask_files = mask_files
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Load image using Rasterio (shape: 2, 256, 256) and mask (256, 256)
        image = np.load(self.image_files[idx])
        mask = np.load(self.mask_files[idx])

        # Add channel dimension to mask -> (1, 256, 256)
        mask = np.expand_dims(mask, axis=0)

        data_dict = {"image": image, "label": mask}

        # Apply MONAI transform
        if self.transform:
            data_dict = self.transform(data_dict)

        return data_dict

def get_train_transform():
    """
    Return a MONAI composition of augmentations to reach IoU > 0.85
    """
    return Compose([
        # Data augmentation
        RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
        RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
        RandRotate90d(keys=['image', 'label'], prob=0.75, max_k=3),

        # Intensity Augmentation (Handles atmospheric variance)
        RandGaussianNoised(keys=['image'], prob=0.2, mean=0.0, std=0.01),
        RandIntensityDistortiond(keys=['image'], prob=0.2),

        # Convert to PyTorch Tensors
        ToTensord(keys =['image', 'label'])
    ])


In [ ]:
## 2.
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from monai.losses import DiceCELoss #Combination of Dice and Cross Entropy
from pathlib import Path
# from src.models.attention_unet import AttentionUNet
# from src.models.dataset import ForestCarbonDataset, get_train_transform

# Infrastructure setup (Download once to local disk)
def download_training_data(bucket_name, local_root_dir):
  """
  Download processed .npy chips from S3 to local storage for speed
  """
  s3 = boto3.resource("s3")
  bucket = s3.Bucket(bucket_name)

  if not os.path.exists(local_root_dir):
    os.makedirs(local_root_dir)

  print(f"Downloading data from s3://{bucket_name}/processed/ to {local_root_dir}...")
  count = 0
  for obj in bucket.objects.filter(Prefix="processed/"):
    local_path = os.path.join(local_root_dir, obj.key)
    local_dir = os.path.dirname(local_path)
    if not os.path.exists(local_dir):
      os.makedirs(local_dir)
    bucket.download_file(obj.key, local_path)
    count+=1
    if count % 500 == 0:
      print(f"Downloaded {count} files...")
  print(f"Download complete! Total files: {count}")

def prepare_data_loader(image_paths, mask_paths, batch_size=16, train_split=0.8):
  """
  Creates training and validation DataLoaders
  """
  # 1. Split the data into Train & Validation sets
  dataset_size = len(image_paths)
  train_len= int(dataset_size * train_split)
  val_len = dataset_size - train_len

  # Split the path first to ensure no leakage
  # Use indices to make sure images and masks are always together
  indices = list(range(dataset_size))
  train_indices = indices[:train_len]
  val_indices = indices[train_len:]

  train_images = [image_paths[i] for i in train_indices]
  train_masks = [mask_paths[i] for i in train_indices]
  val_images = [image_paths[i] for i in val_indices]
  val_masks = [mask_paths[i] for i in val_indices]

  #2. Create Dataset object
  # Only the training set gets augmentation (lips, rotation)
  train_ds = ForestCarbonDataset(train_images, train_masks,
      transform=get_train_transform())

  val_ds = ForestCarbonDataset(val_images, val_masks,
    transform=None) # No augmentation on validation set

  #3. Create DataLoaders
  # num_workers=2 allows the CPU to prepare the next batch while the GPU trains
  # pin_memory = True speeds up the transfer from CPU to GPU
  train_loader = DataLoader(
      train_ds,
      batch_size=batch_size,
      shuffle=True,
      num_workers=2,
      pin_memory=True
  )
  val_loader = DataLoader(
      val_ds,
      batch_size=batch_size,
      shuffle=False,
      num_workers=2,
      pin_memory=True)

  return train_loader, val_loader

In [ ]:
# Run download_training_data
download_training_data("forest-carbon-dung-processed", "/content/data")

## Peak at 1 image to confirm both data transfer and the image quality

In [ ]:
def peek_at_processed_data(image_path, mask_path):
  """
  Helper function to visualize a single chip and its mask
  """
  image = np.load(image_path) # Shape (2, 256, 256) -> Red, INR
  mask = np.load(mask_path) # Shape (256, 256)

  fig, ax = plt.subplots(1, 2, figsize=(10, 5))

  # Show the Red band (Channel 0)
  ax[0].imshow(image[0], cmap='Reds')
  ax[0].set_title('Red Band (B04)')

  # Show the NIR band (Channel 1) or the binary masks
  ax[1].imshow(mask, cmap="Greens")
  ax[1].set_title("Forest Mask (Ground Truth)")

  plt.show()

## Start the training! 🥇🔥

In [ ]:
#--- Main training execution ---
import glob

def start_training():
  # HARDWARE CHECK
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  # Hyperparametes
  EPOCHS = 50
  BATCH_SIZE = 16
  LEARNING_RATE = 1e-4

  # Root dir containing all of the downloaded data
  data_root = '/content/data/processed'

  # Find all of the file and masks
  all_image_files = glob.glob(os.path.join(data_root, '**/images/chip_*.npy'), recursive=True)
  all_mask_files = glob.glob(os.path.join(data_root, '**/masks/mask_*.npy'), recursive=True)

  # Sorted to make sure consistent order of files and their masks
  all_image_files = sorted(all_image_files)
  all_mask_files = sorted(all_mask_files)

  print(f"Found {len(all_image_files)} images and {len(all_mask_files)} masks.")

  # Make sure the number are the same
  assert len(all_image_files) == len(all_mask_files), "Mismatch between images and masks"

  # Create train and val_loader
  train_loader, val_loader = prepare_data_loader(all_image_files, all_mask_files, batch_size=BATCH_SIZE)

  # Initialize model
  model = AttentionUNet(img_ch=2, output_ch=1).to(device)

  # Loss and Optimizer
  loss_function = DiceCELoss(sigmoid=True)
  optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

  # Training loop
  print(f"Starting training on {device}...")
  for epoch in range(EPOCHS):
      model.train()
      step = 0
      for batch_data in train_loader:
          inputs = batch_data['image'].to(device)
          labels= batch_data['label'].to(device)

          optimizer.zero_grad()
          outputs = model(inputs)
          loss = loss_function(outputs, labels)
          loss.backward()
          optimizer.step()

          if step % 100 ==0:
            print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")
          step+=1

      # Save checkpoint to S3 after each epoch
      checkpoint_path = f"model_epoch_{epoch}.pth"
      torch.save(model.state_dict(), checkpoint_path)
      # Add core here to upload .pth to S3 "result" buckets
      s3_results = boto3.client('s3')
      s3_key = f"checkpoints/model_epoch_{epoch}.pth"
      s3_results.upload_file(checkpoint_path, "forest-carbon-dung-results", s3_key)
      print(f"Uploaded checkpoint to s3://forest-carbon-dung-results/{s3_key}")

def get_iou(y_pred, y_true, threshold=0.5):
    """
    Calculate the Intersection over Union (IoU) metric.
    """
    y_pred = (y_pred > threshold).float() # Binary predictions
    intersection = (y_pred * y_true).sum()
    union = y_pred.sum() + y_true.sum() - intersection

    # Avoid division by zero
    if union == 0:
        return 1.0
    return (intersection / union).item()

def get_dice_score(y_pred, y_true, threshold=0.5):
    """
    Calculate the Dice Coefficient (F1-score for pixels)
    """
    y_pred = (y_pred > threshold).float()
    intersection = (y_pred * y_true).sum()
    total = y_pred.sum() + y_true.sum()

    if total == 0:
        return 1.0
    return (2. * intersection / total).item()


if __name__ == "__main__":
    start_training()

## Evaluating the model
1. Load the best weights from S3
2. Run the prediction on test set
3. Calculate the average IoU and Dice Score

In [ ]:
from typing_extensions import final
from monai.metrics import compute_iou, compute_meandice

def evaluate_model(model, test_loader, device, checkpoint_path=None):
  """
  Final evaluation on the Test set to verfiy the 0.85 IoU milestones
  """
  #1 Load the best weight on the Test set to verify the 0.85 IoU milestone
  if checkpoint_path:
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"Loaded model from {checkpoint_path}")

  model.eval()
  all_iou = []
  all_dice =[]

  with torch.no_grad():
    for batch_data in test_loader:
      inputs= batch_data['image'].to(device)
      labels = batch_data['label'].to(device)

      # Inference
      outputs = model(inputs)

      # Post-processing: Apply sigmoid and threshold at 0.5
      preds= (torch.sigmoid(outputs) > 0.5).float()

      # Calculate metrics using MONAI's optimized function
      # ignore_empty=False to be strict with our 0.85 goal
      iou_score = compute_iou(y_pred=preds, y=labels, ignore_empty=False)
      dice_score = compute_meandice(y_pred=preds, y=labels, ignore_empty=False)

      all_iou.append(torch.mean(iou_score).item())
      all_dice.append(torch.mean(dice_score).item())
  final_iou = np.mean(all_iou)
  final_dice = np.mean(all_dice)

  print("\n" +"="*30)
  print(f"Final Metrics for Portfolio:")
  print(f"Target IoU: > 0.85")
  print(f"Final IoU: {final_iou:.4f} {"✅MET" if final_iou > 0.85 else "⛔NOT MET"}")
  print(f"Final Dice: {final_dice:.4f}")

  return final_iou, final_dice